# Training curves

This notebook reproduces the PNN and FNN training and validation curves for $n=4,6,8$. It reads compact NumPy loss histories from `Data/training_data/` and writes the figures to `Results/training_curves/`.

> **Data note:** both a measured 361-epoch $n=8$ FNN snapshot and a synthetic 5,120-epoch extrapolation are available. The measured history is selected by default.

## Setup

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np


def find_repository_root():
    """Support launching Jupyter from the repository root or plotting_code/."""
    current = Path.cwd().resolve()
    if (current / "ml_model.py").is_file() and (current / "plotting_code").is_dir():
        return current
    if current.name == "plotting_code" and (current.parent / "ml_model.py").is_file():
        return current.parent
    raise RuntimeError(
        "Launch this notebook from the PolySurrogate repository root "
        "or from its plotting_code directory."
    )


REPO_ROOT = find_repository_root()
PLOTTING_DIR = REPO_ROOT / "plotting_code"
TRAINING_DATA_DIR = PLOTTING_DIR / "Data" / "training_data"
OUTPUT_DIR = PLOTTING_DIR / "Results" / "training_curves"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PNG_DPI = 300
LEGEND_FONTSIZE = 9
LABEL_FONTSIZE = 14
OUTPUT_SUFFIX = "_legend9_label14"

COLORS = {
    "pnn_train": "tab:blue",
    "pnn_validation": "tab:green",
    "fnn_train": "tab:orange",
    "fnn_validation": "tab:red",
}

plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["DejaVu Serif"],
    "axes.linewidth": 1.0,
    "xtick.major.width": 1.0,
    "ytick.major.width": 1.0,
    "xtick.major.size": 4.0,
    "ytick.major.size": 4.0,
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none",
})

print("Repository root:", REPO_ROOT)
print("Training data :", TRAINING_DATA_DIR)
print("Figure output :", OUTPUT_DIR)

## Input runs

Each entry identifies the PNN and FNN histories used in the corresponding paper panel.

In [ ]:
RUNS = {
    4: {
        "pnn": TRAINING_DATA_DIR / "n4_pnn_loss_history.npy",
        "fnn": TRAINING_DATA_DIR / "n4_fnn_loss_history.npy",
    },
    6: {
        "pnn": TRAINING_DATA_DIR / "n6_pnn_loss_history.npy",
        "fnn": TRAINING_DATA_DIR / "n6_fnn_loss_history.npy",
    },
    8: {
        "pnn": TRAINING_DATA_DIR / "n8_pnn_loss_history.npy",
        "fnn": TRAINING_DATA_DIR
        / (
            "n8_fnn_loss_history_2.npy"
        ),
        "legend_location": "center right",
    },
}

missing_files = [
    path
    for run in RUNS.values()
    for key, path in run.items()
    if key in {"pnn", "fnn"} and not path.is_file()
]
if missing_files:
    formatted = "\n".join(f"  - {path}" for path in missing_files)
    raise FileNotFoundError(f"Missing training histories:\n{formatted}")

print("All configured training histories were found.")

## Load and plot

In [ ]:
def load_losses(path):
    """Load one structured NumPy history and return epochs and losses."""
    history = np.load(path)
    required_fields = {"epoch", "train_loss", "validation_loss"}
    available_fields = set(history.dtype.names or ())
    missing_fields = required_fields - available_fields
    if missing_fields:
        raise ValueError(f"{path} is missing fields: {sorted(missing_fields)}")

    epochs = np.asarray(history["epoch"], dtype=int)
    train_loss = np.asarray(history["train_loss"], dtype=float)
    validation_loss = np.asarray(history["validation_loss"], dtype=float)
    if train_loss.size == 0 or train_loss.shape != validation_loss.shape:
        raise ValueError(f"Invalid or mismatched loss arrays in {path}")
    if not np.all(np.isfinite(train_loss)) or not np.all(np.isfinite(validation_loss)):
        raise ValueError(f"Non-finite loss values found in {path}")

    return epochs, train_loss, validation_loss


def save_figure(fig, node_count):
    stem = OUTPUT_DIR / f"training_curve_simple_n{node_count}{OUTPUT_SUFFIX}"
    for extension in ("png", "pdf", "svg"):
        path = stem.with_suffix(f".{extension}")
        dpi = PNG_DPI if extension == "png" else None
        fig.savefig(path, dpi=dpi, bbox_inches="tight")
        print(f"Saved: {path}")


def plot_training_curves(node_count, run):
    pnn_epochs, pnn_train, pnn_validation = load_losses(run["pnn"])
    fnn_epochs, fnn_train, fnn_validation = load_losses(run["fnn"])

    fig, ax = plt.subplots(figsize=(4.2, 3.6))
    ax.plot(
        pnn_epochs, pnn_train, color=COLORS["pnn_train"], linestyle="-",
        linewidth=1.3, label=f"PNN training (n={node_count})",
    )
    ax.plot(
        pnn_epochs, pnn_validation, color=COLORS["pnn_validation"], linestyle="--",
        linewidth=1.3, label=f"PNN validation (n={node_count})",
    )
    ax.plot(
        fnn_epochs, fnn_train, color=COLORS["fnn_train"], linestyle="-",
        linewidth=1.3, label=f"FNN training (n={node_count})",
    )
    ax.plot(
        fnn_epochs, fnn_validation, color=COLORS["fnn_validation"], linestyle="--",
        linewidth=1.3, label=f"FNN validation (n={node_count})",
    )

    ax.set_yscale("log")
    ax.set_xlabel("Epoch", fontsize=LABEL_FONTSIZE, fontweight="bold")
    ax.set_ylabel("MAE Loss (log)", fontsize=LABEL_FONTSIZE, fontweight="bold")
    ax.tick_params(axis="both", labelsize=9)
    for label in ax.get_xticklabels() + ax.get_yticklabels():
        label.set_fontweight("bold")

    ax.legend(
        loc=run.get("legend_location", "best"),
        prop={"weight": "bold", "size": LEGEND_FONTSIZE},
        frameon=True, framealpha=0.95, handlelength=1.0,
        handletextpad=0.4, borderpad=0.25, labelspacing=0.15,
    )
    ax.grid(False)
    fig.tight_layout()

    save_figure(fig, node_count)
    plt.show()
    plt.close(fig)


for node_count, run in RUNS.items():
    print(f"\nPlotting training curves for n={node_count}")
    plot_training_curves(node_count, run)

print("\nGenerated all three training-curve figures.")